In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
criminals_path = '/content/drive/MyDrive/criminals/rajasthanmetadata/'

# Check if it exists
if os.path.exists(criminals_path):
    print("✓ Found criminals folder!")
    files = os.listdir(criminals_path)
    print(f"  Contains {len(files)} files")
else:
    print("✗ Folder not found. Creating it...")
    os.makedirs(criminals_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Found criminals folder!
  Contains 5000 files


In [3]:
!pip install transformers

In [4]:
!pip install -U bitsandbytes
!pip install qwen_vl_utils
!pip install pandas
!pip install  torchvision

In [5]:
import torch
import sklearn
from torch import nn
from torchvision import transforms
from PIL import Image
from transformers import BitsAndBytesConfig,Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
model_name = "Qwen/Qwen3-VL-8B-Instruct"
model_qwen = Qwen3VLForConditionalGeneration.from_pretrained(
        model_name,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
min_pixels = 256 * 28 * 28
max_pixels = 1280 * 28 * 28
processor_qwen = AutoProcessor.from_pretrained(
   model_name , min_pixels=min_pixels, max_pixels=max_pixels
)
print(processor_qwen.__dict__ )
model_qwen.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

{'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'image_token_id': 151655, 'video_token_id': 151656, 'chat_template': '{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0].role == \'system\' %}\n        {%- if messages[0].content is string %}\n            {{- messages[0].content }}\n        {%- else %}\n            {%- for content in messages[0].content %}\n                {%- if \'text\' in content %}\n                    {{- content.text }}\n                {%- endif %}\n            {%- endfor %}\n        {%- endif %}\n        {{- \'\\n\\n\' }}\n    {%- endif %}\n    {{- "# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [10]:
text_prompts = [
    "a man with a beard",
    "a clean shaven man",
    "a man with a moustache",
    "a man without a moustache",
    "a person with a round face",
    "a person with a slim face",
    "a messy looking person",
    "a well groomed person"
]

face_dictionary = {
    "a man with a beard":[],
    "a clean shaven man":[],
    "a man with a moustache":[],
    "a man without a moustache":[],
    "a person with a round face":[],
    "a person with a slim face":[],
    "a messy looking person":[],
    "a well groomed person":[]
}

In [11]:
import re
def preprocess_text(result):
# Your original text


# Option 1: Get the matched text and convert to lowercase
  matches = re.search(r'assistant\s*\n([\s\S]*)',result, re.IGNORECASE)


  if matches:
    # Group 1 contains the text after "assistant"
    final_answer = matches.group(1).strip().lower()  # .group(1) extracts the captured part
  else:
    final_answer = "none"
  return final_answer

In [12]:
image_info=[]
for i in range(0,5000):
  test_img_path = f"/content/drive/MyDrive/criminals/rajasthanmetadata/{i:05}.jpg"
  image = Image.open(test_img_path)
  image = image.resize((224, 224), Image.BICUBIC)
  image = image.convert("RGB")
  print(f"{i:05}.jpg is done")
  system_prompt = '''You are a visual analysis assistant specialized in accurately identifying physical attributes from images. You provide clear, objective, and concise answers.'''

  user_prompt = f'''Analyze the person in the image and answer the following questions about their appearance. Answer each with only "yes" or "no".

1. Does the person have a beard?
2. Is the person without a beard?
3. Does the person have a moustache?
4. Is the person without a moustache?
5. Does the person have a round face?
6. Does the person have a slim face?
7. Does the person look messy/unkempt?
8. Does the person look well-groomed?
Important constraints:
   "round face" and "slim face" are mutually exclusive. They cannot both be "yes" or "no".
   "messy" and "well-groomed" are mutually exclusive. Exactly one must be "yes" or "no".
Format your response as:
beard: [yes/no]
no beard: [yes/no]
moustache: [yes/no]
no moustache: [yes/no]
round face: [yes/no]
slim face: [yes/no]
messy: [yes/no]
well-groomed: [yes/no]'''

  conversation = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_prompt}
        ]
    }
]
  text = processor_qwen.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=True
          )
  image_inputs, video_inputs = process_vision_info(conversation)
  inputs = processor_qwen(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
           )
  inputs = inputs.to("cuda")
  generated_ids = model_qwen.generate(**inputs,return_dict_in_generate=True,
                                         output_scores=True,
                                         do_sample=True,
                                         max_new_tokens=256,
                                         temperature=0.1)
  answer_text = processor_qwen.tokenizer.batch_decode(
    generated_ids.sequences,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

  answer_text = answer_text[0].strip()
  if(i<5):
    print(answer_text)
    print(preprocess_text(answer_text))
  image_info.append(preprocess_text(answer_text))
#


00000.jpg is done
system
You are a visual analysis assistant specialized in accurately identifying physical attributes from images. You provide clear, objective, and concise answers.
user
Analyze the person in the image and answer the following questions about their appearance. Answer each with only "yes" or "no".

1. Does the person have a beard?
2. Is the person without a beard?
3. Does the person have a moustache?
4. Is the person without a moustache?
5. Does the person have a round face?
6. Does the person have a slim face?
7. Does the person look messy/unkempt?
8. Does the person look well-groomed?
Important constraints:
   "round face" and "slim face" are mutually exclusive. They cannot both be "yes" or "no".
   "messy" and "well-groomed" are mutually exclusive. Exactly one must be "yes" or "no".
Format your response as:
beard: [yes/no]
no beard: [yes/no]
moustache: [yes/no]
no moustache: [yes/no]
round face: [yes/no]
slim face: [yes/no]
messy: [yes/no]
well-groomed: [yes/no]
ass

KeyboardInterrupt: 

In [13]:
print(image_info)
print(len(image_info))

['beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: no\nwell-groomed: yes', 'beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: no\nwell-groomed: yes', 'beard: yes\nno beard: no\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: yes\nwell-groomed: no', 'beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: yes\nwell-groomed: no', 'beard: yes\nno beard: no\nmoustache: yes\nno moustache: no\nround face: no\nslim face: yes\nmessy: no\nwell-groomed: yes', 'beard: yes\nno beard: no\nmoustache: yes\nno moustache: no\nround face: no\nslim face: yes\nmessy: yes\nwell-groomed: no', 'beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: yes\nwell-groomed: no', 'beard: no\nno beard: yes\nmoustache: no\nno moustache: yes\nround face: yes\nslim face: no\nmessy: no\nwell-groomed: yes', 'beard:

In [14]:
for i in range(1265,5000):
  test_img_path = f"/content/drive/MyDrive/criminals/rajasthanmetadata/{i:05}.jpg"
  image = Image.open(test_img_path)
  image = image.resize((224, 224), Image.BICUBIC)
  image = image.convert("RGB")
  print(f"{i:05}.jpg is done")
  system_prompt = '''You are a visual analysis assistant specialized in accurately identifying physical attributes from images. You provide clear, objective, and concise answers.'''

  user_prompt = f'''Analyze the person in the image and answer the following questions about their appearance. Answer each with only "yes" or "no".

1. Does the person have a beard?
2. Is the person without a beard?
3. Does the person have a moustache?
4. Is the person without a moustache?
5. Does the person have a round face?
6. Does the person have a slim face?
7. Does the person look messy/unkempt?
8. Does the person look well-groomed?
Important constraints:
   "round face" and "slim face" are mutually exclusive. They cannot both be "yes" or "no".
   "messy" and "well-groomed" are mutually exclusive. Exactly one must be "yes" or "no".
Format your response as:
beard: [yes/no]
no beard: [yes/no]
moustache: [yes/no]
no moustache: [yes/no]
round face: [yes/no]
slim face: [yes/no]
messy: [yes/no]
well-groomed: [yes/no]'''

  conversation = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_prompt}
        ]
    }
]
  text = processor_qwen.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=True
          )
  image_inputs, video_inputs = process_vision_info(conversation)
  inputs = processor_qwen(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
           )
  inputs = inputs.to("cuda")
  generated_ids = model_qwen.generate(**inputs,return_dict_in_generate=True,
                                         output_scores=True,
                                         do_sample=True,
                                         max_new_tokens=256,
                                         temperature=0.1)
  answer_text = processor_qwen.tokenizer.batch_decode(
    generated_ids.sequences,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

  answer_text = answer_text[0].strip()
  if(i<5):
    print(answer_text)
    print(preprocess_text(answer_text))
  image_info.append(preprocess_text(answer_text))
#

01265.jpg is done
01266.jpg is done
01267.jpg is done
01268.jpg is done
01269.jpg is done
01270.jpg is done
01271.jpg is done
01272.jpg is done
01273.jpg is done
01274.jpg is done
01275.jpg is done
01276.jpg is done
01277.jpg is done
01278.jpg is done
01279.jpg is done
01280.jpg is done
01281.jpg is done
01282.jpg is done
01283.jpg is done
01284.jpg is done
01285.jpg is done
01286.jpg is done
01287.jpg is done
01288.jpg is done
01289.jpg is done
01290.jpg is done
01291.jpg is done
01292.jpg is done
01293.jpg is done
01294.jpg is done
01295.jpg is done
01296.jpg is done
01297.jpg is done
01298.jpg is done
01299.jpg is done
01300.jpg is done
01301.jpg is done
01302.jpg is done
01303.jpg is done
01304.jpg is done
01305.jpg is done
01306.jpg is done
01307.jpg is done
01308.jpg is done
01309.jpg is done
01310.jpg is done
01311.jpg is done
01312.jpg is done
01313.jpg is done
01314.jpg is done
01315.jpg is done
01316.jpg is done
01317.jpg is done
01318.jpg is done
01319.jpg is done
01320.jpg 

In [15]:
print(image_info)
print(len(image_info))

['beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: no\nwell-groomed: yes', 'beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: no\nwell-groomed: yes', 'beard: yes\nno beard: no\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: yes\nwell-groomed: no', 'beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: yes\nwell-groomed: no', 'beard: yes\nno beard: no\nmoustache: yes\nno moustache: no\nround face: no\nslim face: yes\nmessy: no\nwell-groomed: yes', 'beard: yes\nno beard: no\nmoustache: yes\nno moustache: no\nround face: no\nslim face: yes\nmessy: yes\nwell-groomed: no', 'beard: no\nno beard: yes\nmoustache: yes\nno moustache: no\nround face: yes\nslim face: no\nmessy: yes\nwell-groomed: no', 'beard: no\nno beard: yes\nmoustache: no\nno moustache: yes\nround face: yes\nslim face: no\nmessy: no\nwell-groomed: yes', 'beard:

In [17]:
import pandas as pd
import csv
with open("data1.csv", "w", newline="") as file:
    writer = csv.writer(file)

    # header
    writer.writerow(["Name","Image_Info"])

    for i in range(5000):  # simulate huge data
        row = [f"{i:05}.jpg",image_info[i].replace("\n", " | ")]
        writer.writerow(row)